# Veo 모델 기반 멀티모달 비디오 생성 및 시네마틱 연출 실습

**💡 사전 필수 준비 (API Key 설정)**
1. Colab 왼쪽 사이드바에서 **'열쇠 아이콘'(보안 비밀)**을 클릭합니다.
2. 새 비밀 추가: 이름에 `GEMINI_API_KEY` 입력, 값에 API 키 붙여넣기
3. **'노트북 액세스' 토글 버튼을 켜서 활성화**합니다.

## 제1장. 강의 개요 및 비디오 생성 환경 구축

In [ ]:
# 1-1. 최신 SDK 설치 (비디오/이미지 처리용)
!pip install -q -U google-genai pillow

import time
from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import Image, Markdown, display, Video
from PIL import Image as PILImage

# 1-2. 보안 비밀(Secret)에서 API Key 로드 및 Client 초기화
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
    print("✅ Gemini API Client 초기화 성공!")
except userdata.SecretNotFoundError:
    print("❌ 에러: 'GEMINI_API_KEY'가 설정되지 않았습니다. 보안 비밀 탭을 확인해주세요.")

# 1-3. 사용할 모델 ID 정의
IMAGE_MODEL_ID = "gemini-3.1-flash-image-preview"
TEXT_MODEL_ID = "gemini-3-flash-preview"
VIDEO_MODEL_ID = "veo-3.1-fast-generate-001"

# (유틸리티) 비디오 출력 함수
def show_video(video_bytes, filename):
    with open(filename, "wb") as f:
        f.write(video_bytes)
    display(Video(filename, embed=True, width=600))

## 제2장. 텍스트 기반 기본 비디오 생성 (Text-to-Video)

In [ ]:
print("⏳ [1/4] 기초 영상을 렌더링 중입니다. (약 1~3분 소요)...")

# 2-1. 단순 텍스트 프롬프트로 비디오 생성 요청 (비동기 처리)
operation = client.models.generate_videos(
    model=VIDEO_MODEL_ID,
    prompt="A person explaining the Pythagorean theorem.",
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=8,         # 8초 영상
        resolution="720p",          # 해상도 설정
        person_generation="allow_adult",
        generate_audio=True,        # 오디오 동시 생성
    ),
)

# 2-2. 작업이 완료될 때까지 Polling 대기 (15초 간격)
while not operation.done:
    print(".", end="")
    time.sleep(15)
    operation = client.operations.get(operation)

if operation.response:
    print("\n✅ 기초 비디오 렌더링 완료!")
    show_video(operation.result.generated_videos[0].video.video_bytes, "basic_video.mp4")

## 제3장. 연속성 유지를 위한 베이스 프레임 설계 (Image-to-Video 준비)

In [ ]:
print("⏳ [2/4] 베이스 이미지 렌더링 중...")

# 3-1. (실습용) 이전 시간에 만든 강의 슬라이드 더미 이미지 생성
slide_img = PILImage.new('RGB', (800, 450), color = '#eef2f5')
slide_img.save("slide-image.png")

with open("slide-image.png", "rb") as f:
    image_bytes = f.read()

# 3-2. 슬라이드 이미지를 참조하여 '강의실과 교수님'을 추가한 새로운 베이스 프레임 생성
response = client.models.generate_content(
    model=IMAGE_MODEL_ID,
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
        "Generate a professor in a classroom full of students in front of the provided slide image."
    ],
    config=types.GenerateContentConfig(
        response_modalities=['IMAGE'],
        image_config=types.ImageConfig(aspect_ratio="16:9"),
    ),
)

for part in response.candidates[0].content.parts:
    if part.inline_data:
        img_data = part.inline_data.data
        display(Image(data=img_data, width=500))
        # 최종 영상의 입력 소스로 저장
        with open("video-image.png", "wb") as f:
            f.write(img_data)

print("✅ 영상의 첫 시작점(베이스 프레임) 준비 완료!")

## 제4장. 시네마틱 프롬프트 엔지니어링 (Cinematic Prompt Enhancement)

In [ ]:
# 4-1. 영상 연출, 음향, 카메라, 발화 대사 변수화
subject = "a professor"
action = "giving a lecture on the Pythagorean theorem"
scene = "in a classroom, presenting in front of students"
style = "photorealistic"
camera_angle = "eye-level shot"
camera_movement = "zoom in"
sound_effects = "clicking of computer keys"
dialogue = "When you square a side length, like a or b, you are literally calculating the area of a square built perfectly off that side."

keywords = [subject, action, scene, style, camera_angle, camera_movement, sound_effects, dialogue]

# 4-2. LLM 감독(Director) 에이전트를 통한 프롬프트 자동 고도화 합성
gemini_prompt = f"""
Your task is to expand the following keywords into a single, high-fidelity, 
descriptive prompt for video generation. Every single keyword MUST be 
included. Synthesize them into a single, cohesive, and cinematic 
instruction. Do not add any new core concepts. Output ONLY the final 
prompt string, without any introduction or explanation. 

Mandatory Keywords: {",".join(keywords)}
"""

response = client.models.generate_content(
    model=TEXT_MODEL_ID,
    contents=gemini_prompt,
)

video_prompt = response.text
display(Markdown(f"**🎬 완성된 시네마틱 촬영 지시서 (대사 포함):**\n> {video_prompt}"))

## 제5장. 고품질 최종 영상 렌더링 및 실무 응용 (Image-to-Video)

In [ ]:
print("⏳ [4/4] 최종 시네마틱 비디오(오디오 포함) 렌더링 중입니다. (수 분 소요)...")

# 5-1. 베이스 이미지 + 고도화된 프롬프트를 동시에 주입하여 연속성 있는 영상 생성
operation = client.models.generate_videos(
    model=VIDEO_MODEL_ID,
    prompt=video_prompt,
    image=types.Image.from_file(location="video-image.png"), # 3장에서 만든 베이스 프레임 입력
    config=types.GenerateVideosConfig(
        aspect_ratio="16:9",
        number_of_videos=1,
        duration_seconds=8,
        resolution="720p",
        person_generation="allow_adult",
        generate_audio=True, # 프롬프트에 작성한 대사(Dialogue)를 기반으로 입모양을 맞춰 음성 생성
    ),
)

while not operation.done:
    print(".", end="")
    time.sleep(15)
    operation = client.operations.get(operation)

if operation.response:
    print("\n✅ 최종 멀티모달 시네마틱 비디오 렌더링 완료!")
    show_video(operation.result.generated_videos[0].video.video_bytes, "final_cinematic_video.mp4")
    display(Markdown("**🎉 축하합니다! 좌측 폴더 탭에서 `final_cinematic_video.mp4`를 다운로드할 수 있습니다.**"))